# 03 — Transfer learning (EfficientNetB0)

Two-stage training: freeze backbone → train head, then unfreeze top layers and fine-tune at low LR.  
For a full run (all epochs) use `python scripts/train_transfer.py` from `model/`.

This notebook runs **1 epoch per stage** as a smoke template.

In [1]:
import sys
from pathlib import Path


def _add_model_src() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / "model" / "src"
        if candidate.is_dir():
            s = str(candidate)
            if s not in sys.path:
                sys.path.insert(0, s)
            return base
    raise RuntimeError(f"Cannot find model/src from {cwd}. Run Jupyter from the repo root.")


REPO_ROOT = _add_model_src()
print("Repo root:", REPO_ROOT)

Repo root: /Users/sandinosaso/repos/pathsight


In [ ]:
from model_service.config import ModelServiceConfig
from model_service.preprocess.loaders import build_pcam_datasets
from model_service.evaluation.plots import plot_history
from model_service.training.callbacks import default_callbacks
from model_service.training.train import run_training
from model_service.training.transfer_learning import (
    build_efficientnet_model,
    unfreeze_and_finetune,
)

In [3]:
cfg = ModelServiceConfig()
cfg.data.batch_size = 32

# EfficientNet expects its own preprocessing (scales to [-1,1])
train_ds, val_ds, _, _ = build_pcam_datasets(
    cfg, download=True, use_efficientnet_preprocess=True
)
model = build_efficientnet_model(learning_rate=cfg.train.learning_rate)
model.summary(expand_nested=False)

Model: "efficientnetb0_pcam"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,213,668 (16.07 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [4]:
# --- Stage 1: frozen backbone, train head ---
models_dir  = REPO_ROOT / "artifacts" / "models"
metrics_dir = REPO_ROOT / "artifacts" / "metrics"
figures_dir = REPO_ROOT / "artifacts" / "figures"
for d in [models_dir, metrics_dir, figures_dir]:
    d.mkdir(parents=True, exist_ok=True)

cbs1 = default_callbacks(
    models_dir  / "efficientnet_stage1_nb.keras",
    metrics_dir / "efficientnet_stage1_nb.csv",
    monitor="val_auc",
    mode="max",
    early_stopping_patience=cfg.train.early_stopping_patience,
)
h1 = run_training(model, train_ds, val_ds, epochs=4, callbacks=cbs1)
plot_history(h1, figures_dir / "efficientnet_stage1_nb.png")

Epoch 1/4


2026-04-15 16:36:25.539649: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:376] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2026-04-15 16:36:35.530700: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:23: Filling up shuffle buffer (this may take a while): 1796 of 4096
/Users/sandinosaso/repos/pathsight/model/src/model_service/data/stain_normalisation.py:94: RuntimeWarning: overflow encountered in exp
  img_norm = np.exp(-OD_norm) * 255.0
2026-04-15 16:36:45.539954: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:23: Filling up shuffle buffer (this may take a while): 3722 of 4096
2026-04-15 16:36:47.335045: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


 411/8192 ━━━━━━━━━━━━━━━━━━━━ 55:55 431ms/step - accuracy: 0.7464 - auc: 0.8160 - loss: 0.5242 - precision: 0.7440 - recall: 0.7582

/Users/sandinosaso/repos/pathsight/model/src/model_service/data/stain_normalisation.py:94: RuntimeWarning: overflow encountered in multiply
  img_norm = np.exp(-OD_norm) * 255.0


8192/8192 ━━━━━━━━━━━━━━━━━━━━ 4007s 486ms/step - accuracy: 0.7954 - auc: 0.8753 - loss: 0.4431 - precision: 0.7950 - recall: 0.7960 - val_accuracy: 0.8005 - val_auc: 0.8849 - val_loss: 0.4238 - val_precision: 0.8044 - val_recall: 0.7938
Epoch 2/4


...

8192/8192 ━━━━━━━━━━━━━━━━━━━━ 3702s 450ms/step - accuracy: 0.8122 - auc: 0.8929 - loss: 0.4131 - precision: 0.8133 - recall: 0.8105 - val_accuracy: 0.8001 - val_auc: 0.8845 - val_loss: 0.4268 - val_precision: 0.8054 - val_recall: 0.7909
Epoch 3/4


2026-04-15 18:45:02.327609: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:23: Filling up shuffle buffer (this may take a while): 2461 of 4096
2026-04-15 18:45:08.760897: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


8192/8192 ━━━━━━━━━━━━━━━━━━━━ 3659s 445ms/step - accuracy: 0.8191 - auc: 0.8998 - loss: 0.4006 - precision: 0.8212 - recall: 0.8159 - val_accuracy: 0.8053 - val_auc: 0.8932 - val_loss: 0.4153 - val_precision: 0.8457 - val_recall: 0.7465
Epoch 4/4


2026-04-15 19:46:01.227413: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:23: Filling up shuffle buffer (this may take a while): 2026 of 4096
2026-04-15 19:46:10.551838: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


8192/8192 ━━━━━━━━━━━━━━━━━━━━ 3651s 443ms/step - accuracy: 0.8243 - auc: 0.9046 - loss: 0.3915 - precision: 0.8261 - recall: 0.8216 - val_accuracy: 0.8026 - val_auc: 0.8927 - val_loss: 0.4246 - val_precision: 0.8540 - val_recall: 0.7295


In [5]:
# --- Stage 2: unfreeze top 40 layers, fine-tune at low LR ---
model = unfreeze_and_finetune(model, num_layers=40, learning_rate=cfg.train.fine_tune_lr)

cbs2 = default_callbacks(
    models_dir  / "efficientnet_finetune_nb.keras",
    metrics_dir / "efficientnet_stage2_nb.csv",
    monitor="val_auc",
    mode="max",
    early_stopping_patience=cfg.train.early_stopping_patience,
)
h2 = run_training(model, train_ds, val_ds, epochs=5, callbacks=cbs2)
plot_history(h2, figures_dir / "efficientnet_stage2_nb.png")

# Save final model for inference
final = models_dir / "best_model.keras"
model.save(final)
print("Saved:", final)

Epoch 1/5


2026-04-15 23:08:22.381449: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:23: Filling up shuffle buffer (this may take a while): 1920 of 4096
2026-04-15 23:08:33.352597: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


8192/8192 ━━━━━━━━━━━━━━━━━━━━ 4133s 501ms/step - accuracy: 0.7840 - auc: 0.8566 - loss: 0.5018 - precision: 0.7949 - recall: 0.7655 - val_accuracy: 0.8045 - val_auc: 0.8892 - val_loss: 0.4194 - val_precision: 0.8312 - val_recall: 0.7637
Epoch 2/5


2026-04-16 00:17:10.508688: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:23: Filling up shuffle buffer (this may take a while): 2399 of 4096
2026-04-16 00:17:19.438636: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


8192/8192 ━━━━━━━━━━━━━━━━━━━━ 4108s 499ms/step - accuracy: 0.8218 - auc: 0.9004 - loss: 0.4010 - precision: 0.8289 - recall: 0.8110 - val_accuracy: 0.8075 - val_auc: 0.8933 - val_loss: 0.4170 - val_precision: 0.8464 - val_recall: 0.7509
Epoch 3/5


...

8192/8192 ━━━━━━━━━━━━━━━━━━━━ 4129s 502ms/step - accuracy: 0.8340 - auc: 0.9124 - loss: 0.3765 - precision: 0.8404 - recall: 0.8247 - val_accuracy: 0.8091 - val_auc: 0.8960 - val_loss: 0.4140 - val_precision: 0.8494 - val_recall: 0.7510
Epoch 4/5


2026-04-16 02:34:28.050260: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:23: Filling up shuffle buffer (this may take a while): 2798 of 4096
2026-04-16 02:34:33.289815: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


8192/8192 ━━━━━━━━━━━━━━━━━━━━ 4123s 501ms/step - accuracy: 0.8424 - auc: 0.9202 - loss: 0.3602 - precision: 0.8486 - recall: 0.8334 - val_accuracy: 0.8148 - val_auc: 0.8995 - val_loss: 0.4080 - val_precision: 0.8526 - val_recall: 0.7607
Epoch 5/5


2026-04-16 03:43:11.421692: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:23: Filling up shuffle buffer (this may take a while): 2604 of 4096
2026-04-16 03:43:17.509264: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


8192/8192 ━━━━━━━━━━━━━━━━━━━━ 4122s 501ms/step - accuracy: 0.8485 - auc: 0.9254 - loss: 0.3483 - precision: 0.8546 - recall: 0.8400 - val_accuracy: 0.8166 - val_auc: 0.8980 - val_loss: 0.4173 - val_precision: 0.8557 - val_recall: 0.7613
Saved: /Users/sandinosaso/repos/pathsight/artifacts/models/best_model.keras
